<a href="https://www.plus2net.com/python/sqlite.php" target="_blank"><img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net.com" width="180"></a>

# Python SQLite3 — Interactive Practice Notebook

**Tutorial:** [Python SQLite3 Database Tutorial](https://www.plus2net.com/python/sqlite.php)

Work through the cells in order. The examples create a dedicated SQLite database named `plus2net_sqlite_practice.db` in the notebook workspace. In Google Colab, files in the temporary session may disappear when the session ends.

No local development website URLs or credentials are required.

## Quick start: Python sqlite3 example

### Create a database, table, and read a row

The built-in sqlite3 module needs no separate installation. This short example uses an in-memory database, creates a table, inserts one row, reads it, and closes the connection.

In [ ]:
import sqlite3
my_conn = sqlite3.connect(':memory:')
my_conn.execute('CREATE TABLE student (id INTEGER PRIMARY KEY, name TEXT)')
my_conn.execute('INSERT INTO student (name) VALUES (?)', ('John Deo',))
print(my_conn.execute('SELECT id, name FROM student').fetchall())
my_conn.close()

**Expected output:**

```text
[(1, 'John Deo')]
```

## Creating and connecting to a SQLite database

### Connect to a file-based SQLite database

SQLite creates the database file when it does not already exist. Use a dedicated practice database rather than a production file.

In [ ]:
import sqlite3
my_conn = sqlite3.connect('plus2net_sqlite_practice.db')
my_conn.execute('PRAGMA foreign_keys = ON')
print('Database connection ready')

**Expected output:**

```text
Database connection ready
```

## Creating a student table

### Create the student table

Use CREATE TABLE IF NOT EXISTS to avoid an error when the table already exists. INTEGER PRIMARY KEY supplies automatically generated numeric IDs when omitted.

In [ ]:
my_conn.execute('''CREATE TABLE IF NOT EXISTS student (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    class TEXT,
    mark INTEGER CHECK (mark BETWEEN 0 AND 100),
    gender TEXT
)''')
my_conn.commit()
print('Student table ready')

**Expected output:**

```text
Student table ready
```

## Adding records to the table

### Insert sample student records

Use placeholders and executemany() for multiple rows. The example includes recognizable student records from this tutorial; the linked full dataset contains more rows.

**Note:** INSERT OR IGNORE skips an existing record with the same primary key. If you have previously added more records, the count may be higher.

In [ ]:
students = [
    (1, 'John Deo', 'Four', 75, 'female'),
    (2, 'Max Ruin', 'Three', 85, 'male'),
    (3, 'Arnold', 'Three', 55, 'male'),
    (4, 'Krish Star', 'Four', 60, 'female'),
    (34, 'Gain Toe', 'Seven', 69, 'male'),
    (35, 'Rows Noump', 'Six', 88, 'female')
]
my_conn.executemany('''INSERT OR IGNORE INTO student
    (id, name, class, mark, gender)
    VALUES (?, ?, ?, ?, ?)''', students)
my_conn.commit()
print('Sample records available:', my_conn.execute('SELECT COUNT(*) FROM student').fetchone()[0])

**Expected output (example):**

```text
Sample records available: 6 (on a new practice database)
```

## Reading and filtering records

### Display student records

fetchall() returns a list of tuples. ORDER BY keeps the demonstration output predictable.

In [ ]:
rows = my_conn.execute('SELECT id, name, class, mark, gender FROM student ORDER BY id').fetchall()
for row in rows:
    print(row)

**Expected output:**

```text
(1, 'John Deo', 'Four', 75, 'female')
(2, 'Max Ruin', 'Three', 85, 'male')
(3, 'Arnold', 'Three', 55, 'male')
(4, 'Krish Star', 'Four', 60, 'female')
(34, 'Gain Toe', 'Seven', 69, 'male')
(35, 'Rows Noump', 'Six', 88, 'female')
```

### Filter and sort records with WHERE and ORDER BY

Pass search values separately instead of building SQL by concatenating user input.

In [ ]:
minimum_mark = 75
rows = my_conn.execute('''SELECT name, mark FROM student
    WHERE mark >= ? ORDER BY mark DESC, name''', (minimum_mark,)).fetchall()
for row in rows:
    print(row)

**Expected output:**

```text
('Rows Noump', 88)
('Max Ruin', 85)
('John Deo', 75)
```

### Get a single matching row

fetchone() retrieves the first record, or returns None if there is no match.

In [ ]:
row = my_conn.execute(
    'SELECT name, mark FROM student WHERE id = ?', (2,)
).fetchone()
print(row)

**Expected output:**

```text
('Max Ruin', 85)
```

## Inspecting table structure

### View the table structure with PRAGMA

PRAGMA table_info() reports the column position, name, declared type, NOT NULL flag, default value, and primary-key flag.

In [ ]:
for row in my_conn.execute('PRAGMA table_info(student)'):
    print(row)

**Expected output:**

```text
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'class', 'TEXT', 0, None, 0)
(3, 'mark', 'INTEGER', 0, None, 0)
(4, 'gender', 'TEXT', 0, None, 0)
```

### Collect column names as a Python list

Each PRAGMA result contains the column name at index 1.

In [ ]:
columns = [row[1] for row in my_conn.execute('PRAGMA table_info(student)')]
print(columns)

**Expected output:**

```text
['id', 'name', 'class', 'mark', 'gender']
```

## Counting and grouping records

### Count rows using COUNT(*)

COUNT(*) counts the records in a table. Add WHERE when only a subset should be counted.

In [ ]:
total = my_conn.execute('SELECT COUNT(*) FROM student').fetchone()[0]
print('Total student records:', total)

**Expected output (example):**

```text
Total student records: 6 (on a new practice database)
```

### Count students by class

GROUP BY summarizes records across categories.

In [ ]:
rows = my_conn.execute('''SELECT class, COUNT(*) AS total FROM student
    GROUP BY class ORDER BY class''').fetchall()
for row in rows:
    print(row)

**Expected output:**

```text
('Four', 2)
('Seven', 1)
('Six', 1)
('Three', 2)
```

## Inserting, updating and deleting records

### Insert a single row and obtain its ID

Use cursor.lastrowid after INSERT. The name is chosen for the dedicated practice database, not for a live student table.

**Note:** The generated ID can differ if you have run the INSERT example before.

In [ ]:
cursor = my_conn.execute(
    'INSERT INTO student (name, class, mark, gender) VALUES (?, ?, ?, ?)',
    ('New Student', 'Four', 45, 'female')
)
my_conn.commit()
print('Inserted row ID:', cursor.lastrowid)

**Expected output (example):**

```text
Inserted row ID: 36 (with the initial sample records)
```

### Update selected records

Update only matching rows with WHERE; rowcount indicates how many records matched.

In [ ]:
cursor = my_conn.execute(
    'UPDATE student SET mark = ? WHERE name = ?',
    (90, 'Max Ruin')
)
my_conn.commit()
print('Rows updated:', cursor.rowcount)

**Expected output:**

```text
Rows updated: 1
```

### Delete a chosen record

DELETE without WHERE removes every row. This example deletes only the practice row inserted earlier.

In [ ]:
cursor = my_conn.execute(
    'DELETE FROM student WHERE name = ?', ('New Student',)
)
my_conn.commit()
print('Rows deleted:', cursor.rowcount)

**Expected output:**

```text
Rows deleted: 1
```

### Find rows affected using changes()

SQLite changes() reports rows changed by the most recent INSERT, UPDATE, or DELETE statement on this connection.

In [ ]:
changed = my_conn.execute('SELECT changes()').fetchone()[0]
print('Rows affected by previous statement:', changed)

**Expected output:**

```text
Rows affected by previous statement: 1
```

### Handle write errors and roll back

A CHECK constraint rejects invalid marks. The context manager commits valid transactions and rolls back after an exception.

In [ ]:
try:
    with my_conn:
        my_conn.execute(
            'INSERT INTO student (name, mark) VALUES (?, ?)',
            ('Invalid Mark', 150)
        )
except sqlite3.IntegrityError as error:
    print('Rejected invalid mark:', type(error).__name__)

**Expected output:**

```text
Rejected invalid mark: IntegrityError
```

## Listing SQLite database tables

### List database tables

sqlite_master describes database objects. This query shows user-created tables.

In [ ]:
rows = my_conn.execute('''SELECT name FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name''').fetchall()
for row in rows:
    print(row[0])

**Expected output:**

```text
student
```

### Count records in each table safely

Table identifiers cannot be SQL parameters; properly quote database identifiers when composing a statement from table names.

In [ ]:
table_names = [row[0] for row in my_conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
)]
for table_name in table_names:
    quoted_name = '"' + table_name.replace('"', '""') + '"'
    total = my_conn.execute('SELECT COUNT(*) FROM ' + quoted_name).fetchone()[0]
    print(table_name, ':', total)

**Expected output:**

```text
student : 6 (after the insert/update/delete sequence)
```

## Practical SQLite features

### Read rows as dictionaries with sqlite3.Row

Set row_factory to sqlite3.Row when you need access by column name, such as row["name"].

In [ ]:
my_conn.row_factory = sqlite3.Row
row = my_conn.execute('SELECT id, name FROM student WHERE id = ?', (1,)).fetchone()
print(dict(row))
my_conn.row_factory = None

**Expected output:**

```text
{'id': 1, 'name': 'John Deo'}
```

### Use a transaction with SAVEPOINT

SAVEPOINT allows testing a change and undoing it without changing the permanent practice data.

In [ ]:
my_conn.execute('SAVEPOINT practice_demo')
my_conn.execute('UPDATE student SET mark = 99 WHERE id = 1')
print('Temporary mark:', my_conn.execute(
    'SELECT mark FROM student WHERE id = 1').fetchone()[0])
my_conn.execute('ROLLBACK TO SAVEPOINT practice_demo')
my_conn.execute('RELEASE SAVEPOINT practice_demo')
print('Restored mark:', my_conn.execute(
    'SELECT mark FROM student WHERE id = 1').fetchone()[0])

**Expected output:**

```text
Temporary mark: 99
Restored mark: 75
```

### Work with SQLite dates

SQLite stores common date values as ISO-format text and provides functions such as date() for calculations.

In [ ]:
today = my_conn.execute("SELECT date('now')").fetchone()[0]
next_week = my_conn.execute("SELECT date('now', '+7 days')").fetchone()[0]
print('Today:', today)
print('After seven days:', next_week)

**Expected output (example):**

```text
Today: YYYY-MM-DD
After seven days: YYYY-MM-DD (seven days later)
```

## Using SQLAlchemy with SQLite

### Install SQLAlchemy

SQLAlchemy is optional. It is useful when one application works with several SQL databases.

Terminal command from the tutorial:

```bash
pip install sqlalchemy
```

Run the following notebook cell to install the package in the active environment.

In [ ]:
%pip install sqlalchemy

### Connect and run SQL using SQLAlchemy 2.x

Use text() for SQL statements and a context manager for the connection. This reads the practice database created above.

In [ ]:
from sqlalchemy import create_engine, text
engine = create_engine('sqlite:///plus2net_sqlite_practice.db')
with engine.connect() as connection:
    rows = connection.execute(
        text('SELECT name, mark FROM student WHERE mark >= :minimum'),
        {'minimum': 80}
    ).fetchall()
    for row in rows:
        print(row)

**Expected output:**

```text
('Max Ruin', 90)
('Rows Noump', 88)
```

### Handle SQLAlchemy database errors

Catch SQLAlchemyError where an application needs to report errors without exposing connection credentials.

In [ ]:
from sqlalchemy.exc import SQLAlchemyError
try:
    with engine.connect() as connection:
        result = connection.execute(text('SELECT COUNT(*) FROM student'))
        print('Total:', result.scalar_one())
except SQLAlchemyError as error:
    print('Database operation failed:', type(error).__name__)

**Expected output (example):**

```text
Total: 6 (on a new practice database)
```

## Exporting SQLite data for Excel

### Export database records to CSV

The built-in csv module produces a spreadsheet-friendly CSV without extra packages. For .xlsx files, follow the related Excel tutorial.

In [ ]:
import csv
rows = my_conn.execute('SELECT id, name, class, mark, gender FROM student ORDER BY id')
with open('student_export.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['id', 'name', 'class', 'mark', 'gender'])
    writer.writerows(rows)
print('Created student_export.csv')

**Expected output:**

```text
Created student_export.csv
```

## Closing and removing databases

### Close the connection

Close the connection after finishing the database tasks.

In [ ]:
my_conn.close()
print('Database connection closed')

**Expected output:**

```text
Database connection closed
```

### Remove a separate example database file

A SQLite database is stored in a file. Only delete a database when you are sure it is no longer needed. This safely demonstrates deletion on a separate temporary example file.

In [ ]:
from pathlib import Path
example_path = Path('sqlite_delete_example.db')
example_connection = sqlite3.connect(str(example_path))
example_connection.close()
example_path.unlink()
print('Example database file removed')

**Expected output:**

```text
Example database file removed
```

## Further practice

Try these exercises using a fresh connection to the practice database:

1. Select students whose marks are between 60 and 90.
2. Group students by gender and count each group.
3. Use a parameterized UPDATE to change a student's mark.
4. Export only students whose mark is 75 or more to a new CSV file.

Use the linked tutorials for [BETWEEN](https://www.plus2net.com/python/sqlite-between.php), [COUNT](https://www.plus2net.com/python/sqlite.php), [UPDATE](https://www.plus2net.com/python/sqlite-update.php), and [Excel export](https://www.plus2net.com/python/sqlite-excel.php).

## Verify your practice results

Run the assertions below after the examples. They verify that the student records, update, rejected invalid mark, and CSV export behaved as expected.

In [ ]:
from pathlib import Path
with sqlite3.connect('plus2net_sqlite_practice.db') as check_conn:
    rows = check_conn.execute('SELECT COUNT(*) FROM student').fetchone()[0]
    john = check_conn.execute('SELECT mark FROM student WHERE id = 1').fetchone()[0]
    max_mark = check_conn.execute('SELECT mark FROM student WHERE id = 2').fetchone()[0]
    invalid_count = check_conn.execute("SELECT COUNT(*) FROM student WHERE name = 'Invalid Mark'").fetchone()[0]
assert rows >= 6
assert john == 75
assert max_mark == 90
assert invalid_count == 0
assert Path('student_export.csv').exists()
print('All SQLite practice checks passed.')

**Expected output:**

```text
All SQLite practice checks passed.
```

**Continue learning:** [Python SQLite3 Tutorial](https://www.plus2net.com/python/sqlite.php) · [Python tutorials at plus2net.com](https://www.plus2net.com/python/site_map.php) · [plus2net.com](https://www.plus2net.com/)